## 🎲 P.3.5: Probability, Logits & Exponentials
### Conceptual Lecture: Normalizing Logit Scores
An LLM's final layer outputs raw numerical scores called **logits** for all words in its vocabulary. These numbers can be anything (positive, negative, large, small). To select a word probabilistically, we must transform these unbounded numbers into clean, positive percentages that sum to exactly $1.0$ (a probability distribution).

Let's see what happens if we attempt to normalize them by dividing directly.

In [ ]:
logits = [2.0, 1.0, -1.0]
total = sum(logits)

# A naive attempt to generate percentages:
naive_probs = [z / total for z in logits]
print(f"Naive Probabilities: {naive_probs}") # Expected positive values, but got negative (-0.5) for the last word!

#### The Mathematical Necessity
Probability cannot be negative. We need an operator that maps any real number (positive or negative) into a positive value, while strictly preserving their relative size order ($a > b \implies f(a) > f(b)$).

Euler's constant raised to the power of $x$ ($e^x$) has exactly these properties. 
We calculate the exponent of each logit, sum them to get a denominator, and divide each exponent by that sum. This is the **Softmax Function**:
$$\sigma(\mathbf{z})_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

### 🧮 Math-to-Python Translation Key: Softmax Probability
1. **Euler's Constant ($e \approx 2.718$)**:
   - Raising $e$ to the power of $x$ ($e^x$) takes any number (negative or positive) and maps it to a strictly positive value, preserving the size ratio order.
   - *Python Equivalent:* `math.exp(x)`

2. **Softmax Function ($\sigma(\mathbf{z})_i$)**:
   - Take the exponent of every score in the list to make them all positive, sum those exponents together to get a total denominator, and then divide each individual exponent by that sum to get a clean percentage.
   - *Python Equivalent:*
     ```python
     exp_scores = [math.exp(z) for z in scores]
     probs = [exp_val / sum(exp_scores) for exp_val in exp_scores]
     ```

---

In [ ]:
import math

def basic_softmax(scores: list[float]) -> list[float]:
    # Rosetta Stone implementation of Softmax
    exp_scores = [math.exp(z) for z in scores]
    sum_exp = sum(exp_scores)
    return [x / sum_exp for x in exp_scores]

probs = basic_softmax([2.0, 1.0, -1.0])
print(f"Softmax Probabilities: {probs}")
assert abs(sum(probs) - 1.0) < 1e-9, "Sum of probabilities must be exactly 1.0!"
assert all(p > 0 for p in probs), "All probabilities must be positive!"

### 🚀 Active Lab Exercise: Softmax with Temperature Scaling
Generative models use a parameter called **Temperature ($T$)** to control creativity.
We scale the logits by temperature before passing them to Softmax:
$$z'_i = \frac{z_i}{T}$$

- Setting $T = 1.0$ performs default Softmax.
- Setting $T > 1.0$ flattens the distribution (increasing entropy, making choices more random).
- Setting $T \to 0$ spikes the highest score (greedy sampling).

**Your Task:** Implement the complete `softmax_with_temp(logits: list[float], T: float) -> list[float]` function.

In [ ]:
def softmax_with_temp(logits: list[float], T: float) -> list[float]:
    # TODO: Implement Softmax with Temperature Scaling.
    # Note: Safeguard against T = 0 by scaling towards greedy choice if T is extremely small (e.g. T < 1e-5).
    scaled_probs = []
    
    # Your code here:

    
    return scaled_probs

# Verification:
test_logits = [3.0, 1.0, 0.0]

# T = 1.0 (Standard)
p_norm = softmax_with_temp(test_logits, T=1.0)
# T = 2.0 (Flattened / High creativity)
p_hot = softmax_with_temp(test_logits, T=2.0)

print(f"Normal (T=1.0): {p_norm}")
print(f"Creative (T=2.0): {p_hot}")

assert abs(sum(p_norm) - 1.0) < 1e-9
assert abs(sum(p_hot) - 1.0) < 1e-9
# Scaling by T=2 should pull probabilities closer to each other (flatter distribution)
assert p_hot[0] < p_norm[0], "T > 1 should lower the probability spike of the maximum logit."
print("Success! Logit scaling and probability Softmax logic is flawless.")

---
### 🚀 Active Lab Exercise 3: Softmax Numerical Stability (The Overflow Trap)
If we pass large logits (like `1000.0`) to a default Softmax, Python calculates $e^{1000}$, which exceeds standard floating-point limits and crashes with `OverflowError`. To prevent this, professional machine learning implementations subtract the maximum logit value from every score before calculating exponentials:
$$\mathbf{z}'_i = z_i - \max(\mathbf{z})$$

This mathematical identity produces the exact same probabilities, but guarantees numbers never overflow.

**Your Task:** Implement a numerically stable softmax function.

In [ ]:
import math

def stable_softmax(logits: list[float]) -> list[float]:
    # TODO: Implement stable softmax. Subtract the maximum logit from all elements first.
    pass

# Verification:
extreme_logits = [1000.0, 999.0, 998.0]

# Confirming standard math.exp(1000) overflows:
try:
    math.exp(1000.0)
    print("WARNING: Your system didn't overflow?")
except OverflowError:
    print("Standard math.exp(1000.0) crashed with OverflowError, as expected.")

probs = stable_softmax(extreme_logits)
# Expected stable probs: [0.66524096, 0.24472847, 0.09003057]
assert abs(sum(probs) - 1.0) < 1e-9
assert abs(probs[0] - 0.66524096) < 1e-5
print(f"Success! Numerically stable Softmax probabilities generated: {probs}")

---
### 🚀 Active Lab Exercise 4: Top-K Extraction & Logit Masking
To prevent an LLM from sampling completely nonsensical words, we often isolate the Top-K highest scores, and set all other scores to negative infinity ($-\infty$) before calling Softmax.

**Your Task:** Mask an array of logits so that only the highest $K$ scores remain, setting all other scores to `-float('inf')`.

In [ ]:
def top_k_mask(logits: list[float], k: int) -> list[float]:
    # TODO: Implement Top-K logit masking. Keep top K elements as-is; set others to -float('inf').
    # Make sure you don't mutate the original input list!
    pass

# Verification:
logits = [1.2, 0.5, 3.8, -0.4, 2.1]
masked = top_k_mask(logits, k=3)

# Top 3 logits are: 3.8 (idx 2), 2.1 (idx 4), 1.2 (idx 0). Others (0.5, -0.4) are masked.
assert masked == [1.2, -float('inf'), 3.8, -float('inf'), 2.1]
print("Success! Top-K masking matches expectation.")

---
### 🚀 Active Lab Exercise 5: Greedy Sampling (Argmax)
In deterministic 'greedy' text generation, the model simply selects the word with the highest overall probability.

**Your Task:** Write an `argmax(probs)` function that returns the list index of the highest probability value, without importing external libraries.

In [ ]:
def argmax(probs: list[float]) -> int:
    # TODO: Return index of the largest float value in the list.
    pass

# Verification:
probs = [0.12, 0.05, 0.68, 0.15]
best_idx = argmax(probs)
assert best_idx == 2, f"Expected index 2, got {best_idx}"
print("Success! Greedy argmax selector complete.")